In [1]:
import pandas as pd

file_path = "ipl_2025_deliveries - copy.xlsx"

kohli = pd.read_excel(file_path, sheet_name="player - Kohli")
buttler = pd.read_excel(file_path, sheet_name="player - Buttler")

In [2]:
def player_stats(df, player_name):
    
    # Total Runs
    total_runs = df['runs_of_bat'].sum()
    
    # Balls Faced (only legal balls)
    balls = df[df['legal ball'] == 1].shape[0]
    
    # Strike Rate
    sr = (total_runs / balls) * 100 if balls > 0 else 0
    
    # Boundaries
    fours = df[df['runs_of_bat'] == 4].shape[0]
    sixes = df[df['runs_of_bat'] == 6].shape[0]
    
    boundary_runs = fours*4 + sixes*6
    boundary_pct = (boundary_runs / total_runs) * 100 if total_runs > 0 else 0
    
    # Dismissals (exclude Not Out / blanks)
    dismissals = df[(df['wicket_type'] != "Not Out") & (df['wicket_type'].notna())].shape[0]
    
    return {
        "Player": player_name,
        "Runs": total_runs,
        "Balls": balls,
        "Strike Rate": round(sr,2),
        "Boundary %": round(boundary_pct,2),
        "Dismissals": dismissals
    }

In [3]:
kohli_stats = player_stats(kohli, "Virat Kohli")
buttler_stats = player_stats(buttler, "Jos Buttler")

summary = pd.DataFrame([kohli_stats, buttler_stats])
print(summary)

        Player  Runs  Balls  Strike Rate  Boundary %  Dismissals
0  Virat Kohli   657    454       144.71       57.53         466
1  Jos Buttler   538    330       163.03       65.43         337


In [12]:
def phase_analysis(df, player_name):
    
    df_legal = df[df['legal ball'] == 1]
    
    phase_stats = df_legal.groupby('match phase').agg({
        'runs_of_bat': 'sum',
        'legal ball': 'count'
    }).reset_index()
    
    phase_stats['Strike Rate'] = (phase_stats['runs_of_bat'] / phase_stats['legal ball']) * 100
    phase_stats['Player'] = player_name
    
    return phase_stats

In [13]:
kohli_phase = phase_analysis(kohli, "Virat Kohli")
buttler_phase = phase_analysis(buttler, "Jos Buttler")

phase_combined = pd.concat([kohli_phase, buttler_phase])

In [14]:
def run_distribution(df, player_name):
    
    dist = df['runs_of_bat'].value_counts().reset_index()
    dist.columns = ['Runs', 'Count']
    dist['Player'] = player_name
    
    return dist

In [15]:
kohli_dist = run_distribution(kohli, "Virat Kohli")
buttler_dist = run_distribution(buttler, "Jos Buttler")

dist_combined = pd.concat([kohli_dist, buttler_dist])

In [16]:
def dismissal_analysis(df, player_name):
    
    dismissals = df[(df['wicket_type'] != "Not Out") & (df['wicket_type'].notna())]
    
    result = dismissals['wicket_type'].value_counts().reset_index()
    result.columns = ['Type', 'Count']
    result['Player'] = player_name
    
    return result

In [17]:
kohli_dismissal = dismissal_analysis(kohli, "Virat Kohli")
buttler_dismissal = dismissal_analysis(buttler, "Jos Buttler")

dismissal_combined = pd.concat([kohli_dismissal, buttler_dismissal])

In [18]:
summary.to_csv("summary.csv", index=False)
phase_combined.to_csv("phase.csv", index=False)
dist_combined.to_csv("distribution.csv", index=False)
dismissal_combined.to_csv("dismissal.csv", index=False)

In [19]:
print(summary)
print(phase_combined.head())
print(dist_combined.head())
print(dismissal_combined.head())

        Player  Runs  Balls  Strike Rate  Boundary %  Dismissals
0  Virat Kohli   657    454       144.71       57.53         466
1  Jos Buttler   538    330       163.03       65.43         337
  match phase  runs_of_bat  legal ball  Strike Rate       Player
0       Death           38          24   158.333333  Virat Kohli
1      Middle          301         228   132.017544  Virat Kohli
2   Powerplay          318         202   157.425743  Virat Kohli
0       Death          133          71   187.323944  Jos Buttler
1      Middle          330         200   165.000000  Jos Buttler
   Runs  Count       Player
0     1    217  Virat Kohli
1     0    133  Virat Kohli
2     4     66  Virat Kohli
3     2     31  Virat Kohli
4     6     19  Virat Kohli
      Type  Count       Player
0  not out    453  Virat Kohli
1   caught     12  Virat Kohli
2   runout      1  Virat Kohli
0  not out    327  Jos Buttler
1   caught      6  Jos Buttler
